# sorted-computational-graph — ex2: topo-sort a multi-depth shared-leaf compute graph (each node once)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `sorted-computational-graph`. Running the final beacon cell reports progress against the `Backprop: Sorted computation graph` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Sorted computation graph` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`sorted-computational-graph`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "sorted-computational-graph"
DD_SUBTOPIC = "Backprop: Sorted computation graph"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## sorted_computational_graph with shared subgraph — quick refresher

ex1 covered a tree-like compute graph + diamond DAG. The ex2 facet is a MORE-shared subgraph: a single leaf consumed by multiple intermediate nodes, and at multiple depths.

Two invariants that get exercised harder here:
- **Each node appears EXACTLY ONCE in the sorted list**, regardless of how many edges point at it. The DFS topo-sort uses a 'permanent' marker to suppress repeats.
- **Parent-before-child holds in reverse order across ALL edges**, including the long-range edge from end-node to leaf via multiple intermediate consumers.

Without proper marker-set handling, repeated visits to the shared leaf either duplicate it in the output or recurse infinitely. The `perm` set guards both.

### Exercise 2 — topo-sort a multi-depth shared-leaf compute graph (each node once)

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply topological sort over a compute graph with a leaf shared by multiple intermediate consumers at different depths, ensuring each node appears exactly once and parent-before-child holds across all edges in the reverse order.
> Keywords: topo-sort, shared-leaf, dedup, multi-depth-dag
> ```

**KCs targeted:** `sorted-computational-graph`, `parents-dict-by-argidx`

Implement `topological_sort(node, get_children)` and `sorted_computational_graph(tensor)`. Specs are the same as ex1, but the test graph here is HARDER:

```
Compute graph (read top-down — each line is a recipe):
leaf:  a
       b
node:  u = a * b
       v = log(a)       <-- a shared with u
       w = u + v        <-- merges two ancestors of a
       z = w * a        <-- a appears again at depth 1!
end:   z
```

Properties this graph exercises:
- `a` is consumed by FOUR nodes at three distinct depths (depth-3 via u, depth-2 via v, depth-1 directly into z).
- `b` is consumed by exactly one node.
- The DAG fans in at `w` and fans out from `a`.

Your sort must:
1. Visit each node EXACTLY ONCE (no duplicates of `a`).
2. End node `z` is FIRST in the result.
3. For EVERY edge `(parent → child)` in the recipe graph: `pos[child] < pos[parent]` in the result.
4. Cycle detection still works (we'll re-test it).

Implementation: same three-color DFS pattern. The shared-leaf case is what the `perm` set is FOR — it stops the second visit to `a` from duplicating it.

Use `id(...)` for set membership (MiniTensors aren't hashable by value).

In [ ]:
def topological_sort(node, get_children):
    """Return descendants of node in topological order (node LAST).
    Raise ValueError on cycle.
    """
    raise NotImplementedError()


def sorted_computational_graph(tensor):
    """Return MiniTensors in reverse-topological order (end node FIRST)."""
    raise NotImplementedError()


def _test_ex2():
    # --- cycle detection still works ---
    class N:
        def __init__(self, name, *children):
            self.name = name
            self.children = list(children)
        def __repr__(self):
            return f'N({self.name})'

    x = N('x'); y = N('y')
    x.children = [y]; y.children = [x]
    try:
        topological_sort(x, lambda n: n.children)
    except ValueError:
        pass
    else:
        raise AssertionError('cycle should raise ValueError')

    # --- build the multi-depth shared-leaf graph ---
    a = MiniTensor(t.tensor([2.0]), requires_grad=True)
    b = MiniTensor(t.tensor([3.0]), requires_grad=True)
    u = MiniTensor(a.array * b.array, requires_grad=True)
    u.recipe = Recipe(func=t.multiply, args=(a.array, b.array), kwargs={}, parents={0: a, 1: b})
    v = MiniTensor(t.log(a.array), requires_grad=True)
    v.recipe = Recipe(func=t.log, args=(a.array,), kwargs={}, parents={0: a})
    w = MiniTensor(u.array + v.array, requires_grad=True)
    w.recipe = Recipe(func=t.add, args=(u.array, v.array), kwargs={}, parents={0: u, 1: v})
    z = MiniTensor(w.array * a.array, requires_grad=True)
    z.recipe = Recipe(func=t.multiply, args=(w.array, a.array), kwargs={}, parents={0: w, 1: a})

    order = sorted_computational_graph(z)

    # --- every node appears EXACTLY ONCE (despite a being shared by u, v, z) ---
    ids = [id(n) for n in order]
    from collections import Counter
    counts = Counter(ids)
    for label, node in [('a', a), ('b', b), ('u', u), ('v', v), ('w', w), ('z', z)]:
        assert counts[id(node)] == 1, (
            f'{label} appears {counts[id(node)]} times in topo sort (expected 1) '
            f'— shared-leaf dedup broke'
        )

    # --- exactly 6 nodes, no extras ---
    assert len(order) == 6, f'expected 6 nodes, got {len(order)}: {[id(n) for n in order]}'
    assert set(ids) == {id(a), id(b), id(u), id(v), id(w), id(z)}

    # --- end node first ---
    assert order[0] is z, f'first should be z, got {order[0]}'

    # --- parent-before-child in reverse-topo: pos[child] < pos[parent] for every edge ---
    pos = {id(n): i for i, n in enumerate(order)}
    edges = [
        (z, w), (z, a),   # z's parents
        (w, u), (w, v),   # w's parents
        (u, a), (u, b),   # u's parents
        (v, a),           # v's parents
    ]
    for child, parent in edges:
        assert pos[id(child)] < pos[id(parent)], (
            f'reverse-topo violated for edge ({child.recipe.func.__name__ if child.recipe else "leaf"})'
            f'@{pos[id(child)]} → {parent}@{pos[id(parent)]}'
        )

    # --- `a` is the deepest-shared node: should appear AFTER u, v, AND z ---
    # (z → a edge, u → a edge, v → a edge — all three say a comes later.)
    assert pos[id(a)] > pos[id(u)], 'a after u (z → w → u → a)'
    assert pos[id(a)] > pos[id(v)], 'a after v (z → w → v → a)'
    assert pos[id(a)] > pos[id(z)], 'a after z (z → a direct)'

    # --- singleton graph (just a leaf) still works ---
    lonely = MiniTensor(t.tensor([5.0]), requires_grad=True)
    order_lone = sorted_computational_graph(lonely)
    assert order_lone == [lonely], f'singleton: {order_lone}'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def topological_sort(node, get_children):
    result = []
    perm = set()   # fully processed nodes — keyed by id()
    temp = set()   # on the DFS stack — cycle detector

    def visit(cur):
        cid = id(cur)
        if cid in perm:
            return
        if cid in temp:
            raise ValueError(f'Cycle at {cur!r} — graph is not a DAG')
        temp.add(cid)
        for child in get_children(cur):
            visit(child)
        temp.remove(cid)
        perm.add(cid)
        result.append(cur)

    visit(node)
    return result


def sorted_computational_graph(tensor):
    def get_parents(t_):
        if t_.recipe is None:
            return []
        return list(t_.recipe.parents.values())
    return topological_sort(tensor, get_parents)[::-1]
```

**Why the perm set is necessary (not just temp).** A two-color DFS (visited / unvisited) would either: revisit shared leaves and duplicate them in the output, OR mark them after first visit but have no way to distinguish 'finished this subtree' from 'currently processing.' The perm/temp split handles BOTH the shared-leaf dedup case (perm) AND the cycle detection (temp).

**Why `a` ends up at the BACK of the reverse-topo result.** `a` has three parent edges pointing INTO it (from u, v, z) — in the forward direction, `a` is the deepest dependency. After reversal, the deepest dependencies come last. The position-invariant test pins this down: `pos[a]` must exceed `pos[u]`, `pos[v]`, AND `pos[z]`.

**Why id(...) and not the object itself.** MiniTensors compare by identity by default (we didn't override `__eq__`), so `set(...)` of MiniTensors works — but for safety, `id(...)` is robust to any future change in equality semantics. PyTorch's autograd graph uses identity keys for the same reason.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()